[메타클래스](https://dojang.io/mod/page/view.php?id=2468)는 클래스를 만드는 클래스인데, 이 메타클래스를 구현하는 방법은 두 가지가 있다.
1. type을 사용하여 동적으로 클래스를 생성하는 방식
1. type을 상속받아서 메타클래스를 구현하는 방식

In [ ]:
Hello = type('Hello', (), {})
print(Hello)
h = Hello()
print(h)

메타클래스를 활용하면 아래와 같은 것들이 가능하다.
1. 클래스 계층 설계할 때 스타일을 강제
1. 메소드를 오버라이드 하도록 요청
1. 클래스 애트리뷰트 사이에 엄격한 관계를 가지도록 요구

In [ ]:
class Meta(type):
    def __new__(meta, name, bases, class_dict):
        print(f'* 실행: {name}의 메타 {meta}.__new__')
        print('기반클래스들:', bases)
        print(class_dict)
        return type.__new__(meta, name, bases, class_dict)

class MyClass(metaclass=Meta):
    stuff = 123

    def foo(self):
        pass

class MySubclass(MyClass):
    other = 567

    def bar(self):
        pass

다각형 클래스 계층 구조의 기반 클래스로 메타클래스를 활용할 수 있다.

In [ ]:
class ValidatePolygon(type):
    def __new__(meta, name, bases, class_dict):
        # Polygon 클래스의 하위 클래스만 검증한다
        if bases:
            if class_dict['sides'] < 3:
                raise ValueError('다각형 변은 3개 이상이어야 함')
        return type.__new__(meta, name, bases, class_dict)

class Polygon(metaclass=ValidatePolygon):
    sides = None # 하위 클래스는 이 애트리뷰트에 값을 지정해야 한다
    
    @classmethod
    def interior_angles(cls):
        return (cls.sides - 2) * 180

class Triangle(Polygon):
    sides = 3

class Rectangle(Polygon):
    sides = 4

class Nonagon(Polygon):
    sides = 9

assert Triangle.interior_angles() == 180
assert Rectangle.interior_angles() == 360
assert Nonagon.interior_angles() == 1260

클래스 정의문 본문이 실행된 직후 예외를 발생시킨다. 즉, 유효하지 않은 클래스를 정의하면 프로그램이 아예 시작되지도 않는다는 뜻이다.

In [ ]:
print('class 이전')

class Line(Polygon):
    print('sides 이전')
    sides = 2
    print('sides 이후')

print('class 이후')

파이썬 3.6부터는 메타클래스를 정의하지 않고 같은 동작을 구현할 수 있는 더 단순한 구문 `__init_subclass__`이 추가되었다.

In [ ]:
class BetterPolygon:
    sides = None  # 하위클래스에서 이 애트리뷰트의 값을 지정해야 함

    def __init_subclass__(cls):
        super().__init_subclass__()
        if cls.sides < 3:
            raise ValueError('다각형 변은 3개 이상이어야 함')

    @classmethod
    def interior_angles(cls):
        return (cls.sides - 2) * 180

class Hexagon(BetterPolygon):
    sides = 6

assert Hexagon.interior_angles() == 720

표준 파이썬 메타클래스의 또 다른 문제점은 하나의 메타클래스만 지정 가능하다는 점이다.

In [ ]:
class ValidateFilled(type):
    def __new__(meta, name, bases, class_dict):
        # Filled 클래스의 하위 클래스만 검증한다
        if bases:
            if class_dict['color'] not in ('red', 'green'):
                raise ValueError('지원하지 않는 color 값')
        return type.__new__(meta, name, bases, class_dict)

class Filled(metaclass=ValidateFilled):
    color = None  # 모든 하위 클래스에서 이 애트리뷰트의 값을 지정해야 한다

class RedPentagon(Filled, Polygon):
    color = 'red'
    sides = 5

해결하기 위해 메타클래스 type 정의를 복잡한 계층으로 설계해야 한다.

In [ ]:
class ValidatePolygon(type):
    def __new__(meta, name, bases, class_dict):
        # 루트 클래스가 아닌 경우만 검증한다
        if not class_dict.get('is_root'):
            if class_dict['sides'] < 3:
                raise ValueError('다각형 변은 3개 이상이어야 함')
        return type.__new__(meta, name, bases, class_dict)

class Polygon(metaclass=ValidatePolygon):
    is_root = True
    sides = None  # 하위 클래스에서 이 애트리뷰트 값을 지정해야 한다

class ValidateFilledPolygon(ValidatePolygon):
    def __new__(meta, name, bases, class_dict):
        # 루트 클래스가 아닌 경우만 검증한다
        if not class_dict.get('is_root'):
            if class_dict['color'] not in ('red', 'green'):
                raise ValueError('지원하지 않는 color 값')
        return super().__new__(meta, name, bases, class_dict)

class FilledPolygon(Polygon, metaclass=ValidateFilledPolygon):
    is_root = True
    color = None  # 하위 클래스에서 이 애트리뷰트 값을 지정해야 한다

class GreenPentagon(FilledPolygon):
    color = 'green'
    sides = 5

greenie = GreenPentagon()
assert isinstance(greenie, Polygon)

# 색 오류
class OrangePentagon(FilledPolygon):
    color = 'orange'
    sides = 5


하지만 이런 접근 방법은 `합성성(composability)`를 해친다. `ValidateFilledPolygon`에 있는 색 검증 로직을 다른 클래스 계층 구조에 적용하려면 모든 로직을 중복 정의해야 한다.

`__init_subclass__` 특별 클래스 메서드를 사용하면 이 문제를 해결할 수 있다.

In [ ]:
class Filled:
    color = None  # 하위 클래스에서 이 애트리뷰트 값을 지정해야 한다

    def __init_subclass__(cls):
        super().__init_subclass__()
        if cls.color not in ('red', 'green', 'blue'):
            raise ValueError('지원하지 않는 color 값')

class RedTriangle(Filled, Polygon):
    color = 'red'
    sides = 3

ruddy = RedTriangle()
assert isinstance(ruddy, Filled)
assert isinstance(ruddy, Polygon)

변의 수를 잘못 지정하거나, 색을 잘못 지정하면 오류가 발생한다.

In [ ]:
print('class 이전')

class BlueLine(Filled, Polygon):
    color = 'blue'
    sides = 2

print('class 이후')

In [ ]:
print('class 이전')

class BeigeSquare(Filled, Polygon):
    color = 'beige'
    sides = 4

print('class 이후')

다이아몬드 상속까지도 지원한다.

In [ ]:
class Top:
    def __init_subclass__(cls):
        super().__init_subclass__()
        print(f'{cls}의 Top')

class Left(Top):
    def __init_subclass__(cls):
        super().__init_subclass__()
        print(f'{cls}의 Left')


class Right(Top):
    def __init_subclass__(cls):
        super().__init_subclass__()
        print(f'{cls}의 Right')


class Bottom(Left, Right):
    def __init_subclass__(cls):
        super().__init_subclass__()
        print(f'{cls}의 Bottom')